In [1]:
import os, sys, asyncio, nest_asyncio, json
from dotenv import load_dotenv

# Essential for Windows + Jupyter + Asynchronous Agents
nest_asyncio.apply()
if sys.platform == 'win32':
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    except:
        pass

load_dotenv()
print("Environment Patched and Ready.")

Environment Patched and Ready.


In [2]:

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import socket
import time

def get_dynamic_data(url):
    # Internet check
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=3)
    except OSError:
        print(f"Network Error: Cannot reach {url}. Check your connection.")
        return None

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        print(f"Selenium is fetching: {url}")
        driver.get(url)

        # time.sleep(5) ki jagah smart wait
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
        except:
            pass

        return driver.page_source

    except Exception as e:
        print(f"⚠️ Scraping Error for {url}: {e}")
        return None

    finally:
        if 'driver' in locals():
            driver.quit()

Data loading

In [3]:
from langchain_community.document_loaders import PyPDFDirectoryLoader, DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from concurrent.futures import ThreadPoolExecutor
from bs4 import BeautifulSoup
import os

# Exact paths 
DATA_DIR = "../data"
PDF_PATH = os.path.join(DATA_DIR, "pdf")
TEXT_FILES_PATH = os.path.join(DATA_DIR, "text_files")
URLS_FILE_PATH = os.path.join(DATA_DIR, "urls.txt")

async def assemble_knowledge_base():
    all_docs = []
    
    # 1. Load PDFs
    if os.path.exists(PDF_PATH) and os.listdir(PDF_PATH):
        pdf_loader = PyPDFDirectoryLoader(PDF_PATH)
        all_docs.extend(pdf_loader.load())
        print(f"PDFs Loaded from {PDF_PATH}")

    # 2. Load Text Files
    if os.path.exists(TEXT_FILES_PATH) and os.listdir(TEXT_FILES_PATH):
        text_loader = DirectoryLoader(TEXT_FILES_PATH, glob="./*.txt", loader_cls=TextLoader)
        all_docs.extend(text_loader.load())
        print(f"Text Files Loaded from {TEXT_FILES_PATH}")

    # 3. Live Web Scraping (Parallel)
    if os.path.exists(URLS_FILE_PATH):
        with open(URLS_FILE_PATH, "r") as f:
            urls = [line.strip() for line in f if line.strip()]
        
        print(f"Scraping {len(urls)} URLs in parallel...")

        with ThreadPoolExecutor(max_workers=3) as executor:
            results = list(executor.map(get_dynamic_data, urls))

        for url, html_content in zip(urls, results):
            if html_content:
                soup = BeautifulSoup(html_content, 'html.parser')
                for element in soup(["script", "style", "nav", "footer", "header"]):
                    element.extract()
                clean_text = soup.get_text(separator="\n", strip=True)
                all_docs.append(Document(
                    page_content=clean_text,
                    metadata={"source": url, "type": "web_live"}
                ))
        print(f"All URLs from urls.txt processed.")

    # 4. Final Hybrid Chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, 
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "]
    )
    chunks = splitter.split_documents(all_docs)
    print(f"TOTAL SEARCHABLE CHUNKS: {len(chunks)}")
    return chunks

# Run the Assembler
raw_chunks = await assemble_knowledge_base()

d:\vector_db\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDFs Loaded from ../data\pdf
Text Files Loaded from ../data\text_files
Scraping 1 URLs in parallel...
Selenium is fetching: https://www.educators.edu.pk/
All URLs from urls.txt processed.
TOTAL SEARCHABLE CHUNKS: 17


to check paths are actually working or not..

In [4]:
import os

# Exact paths based on your actual VS Code project structure
base_data_path = "../data"
paths_to_check = {
    "PDF Folder": os.path.join(base_data_path, "pdf"),
    "Text Files Folder": os.path.join(base_data_path, "text_files"),
    "URLs List": os.path.join(base_data_path, "urls.txt")
}

print(" --- VOCIRA SYSTEM PATH HEALTH CHECK --- \n")

for name, path in paths_to_check.items():
    absolute_path = os.path.abspath(path)
    exists = os.path.exists(path)
    
    status = "FOUND" if exists else "❌ MISSING"
    print(f"{name}: {status}")
    print(f"   Path: {absolute_path}")
    
    if exists:
        if os.path.isdir(path):
            files = os.listdir(path)
            print(f"   Files: {files if files else 'Empty Folder'}")
        else:
            # For files like urls.txt
            with open(path, 'r') as f:
                # Sirf un lines ko count karein jin mein actual URL hai (empty lines ignore)
                valid_lines = [l for l in f.readlines() if l.strip()]
                print(f"   Content: {len(valid_lines)} URLs verified.")
    
    print("-" * 30)

print("\n READINESS: All data paths are verified. Ready for Ingestion.")

 --- VOCIRA SYSTEM PATH HEALTH CHECK --- 

PDF Folder: FOUND
   Path: d:\vector_db\data\pdf
   Files: ['school_policies.pdf']
------------------------------
Text Files Folder: FOUND
   Path: d:\vector_db\data\text_files
   Files: ['general_info.txt']
------------------------------
URLs List: FOUND
   Path: d:\vector_db\data\urls.txt
   Content: 1 URLs verified.
------------------------------

 READINESS: All data paths are verified. Ready for Ingestion.


In [5]:
print(f"I have {len(raw_chunks)} chunks ready for Pinecone.")

I have 17 chunks ready for Pinecone.


## Phase 3: Vector Intelligence & Indexing
In this stage, we transform our processed text chunks into mathematical vectors using the **Hugging Face `all-MiniLM-L6-v2`** embedding model. 

### Key Technical Operations:
1. **Embedding**: Converting text chunks into vectors.
2. **Vector Database**: Connecting to **Pinecone (Serverless)** to store these embeddings.
3. **Indexing**: Mapping the text content and metadata (URLs/PDF sources) to the vectors to enable **Semantic Search**.

*This allows our Agent to perform 'Similarity Search' rather than just keyword matching.*

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from tenacity import retry, stop_after_attempt, wait_exponential
import os
import time

# 1. Initialize the Embedding Model
print("Initializing Embedding Model...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 2. Connect to Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "school-bot-index"
temp_index_name = f"{index_name}-temp"

# 3. Retry wrapper for Pinecone upload
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def upload_to_pinecone(chunks, embeddings, index_name):
    return PineconeVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        index_name=index_name
    )

# 4. Smart Index Management ( upload first, delete after)
if index_name not in pc.list_indexes().names():
    # Fresh start — no old data to worry about
    print(f"Creating new Index: {index_name}...")
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print("New Index is Ready!")

    print(f"Encoding & Uploading {len(raw_chunks)} chunks to Pinecone...")
    vector_store = upload_to_pinecone(raw_chunks, embeddings, index_name)

else:
    # first confirm the new data in temp index then del the old one
    print(f"Existing index found. Starting safe swap...")

    # Step 1: creating Temp index 
    if temp_index_name not in pc.list_indexes().names():
        pc.create_index(
            name=temp_index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        while not pc.describe_index(temp_index_name).status['ready']:
            time.sleep(1)

    # Step 2: Naya data temp mein upload karo
    print(f"Uploading {len(raw_chunks)} chunks to staging index...")
    upload_to_pinecone(raw_chunks, embeddings, temp_index_name)
    print("Upload confirmed. Now replacing old knowledge base...")

    # Step 3: Upload confirm hone KE BAAD hi old delete karo
    pc.delete_index(index_name)
    time.sleep(3)

    # Step 4: Production index dobara banao
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

    # Step 5: Production mein final upload
    vector_store = upload_to_pinecone(raw_chunks, embeddings, index_name)

    # Step 6: Temp index clean up
    pc.delete_index(temp_index_name)
    print("Temp index cleaned up.")

print("=" * 45)
print("STATUS: VOCIRA INTELLIGENCE SYNCED")
print(f"Database updated with PDFs, Text files, and Live Web data.")
print(f"Total Searchable Units: {len(raw_chunks)}")
print("=" * 45)

Initializing Embedding Model...


d:\vector_db\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pakistan\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7938.70it/s]


Existing index found. Starting safe swap...
Uploading 17 chunks to staging index...
Upload confirmed. Now replacing old knowledge base...
Temp index cleaned up.
STATUS: VOCIRA INTELLIGENCE SYNCED
Database updated with PDFs, Text files, and Live Web data.
Total Searchable Units: 17


## Phase 4: The Agentic Reasoning Engine 
Now that our knowledge is vectorized, we initialize the model via Hugging Face.


In [7]:
import os
import time
import httpx
from huggingface_hub import InferenceClient

# 1. Enhanced Retrieval (Fetching Context + Metadata)
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

def school_portal_search(query: str):
    """Fetches text and their sources from Pinecone."""
    try:
        docs = retriever.invoke(query)
        if not docs:
            return "No specific data found in the database.", []
        
        context_parts = []
        sources = set()
        
        for doc in docs:
            context_parts.append(doc.page_content)
            source_info = doc.metadata.get('source', 'Internal Records')
            sources.add(source_info)
            
        return "\n".join(context_parts), list(sources)
    except Exception as e:
        return f"Database Error: {str(e)}", []

# 2. Stable Hugging Face Client
client = InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct",
    token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

# 3. Final Vocira Logic
def ask_vocira(user_query: str):
    print(f"🔍 Searching Knowledge Base...")
    
    context, sources = school_portal_search(user_query)
    
    # Context window guard — prompt won't overflow
    MAX_CONTEXT_CHARS = 3000
    if len(context) > MAX_CONTEXT_CHARS:
        context = context[:MAX_CONTEXT_CHARS] + "\n...[truncated]"
    
    system_prompt = f"""
    You are Vocira, the official AI Assistant for 'The Educators'.
    Use the following verified context to answer the user.
    
    RULES:
    1. Be concise, helpful, and professional.
    2. If the context doesn't contain the answer, politely state that you don't have that specific information.
    3. Strictly avoid making up facts (hallucinations).
    
    CONTEXT:
    {context}
    """
    
    for attempt in range(3):
        try:
            response = client.chat_completion(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_query}
                ],
                max_tokens=1000,
                temperature=0.1
            )
            
            answer = response.choices[0].message.content
            
            if sources:
                source_list = "\n📍 Sources: " + ", ".join(sources)
                return f"{answer}\n{source_list}"
            return answer

        # catching the correct exceptions
        except httpx.HTTPStatusError as e:
            if e.response.status_code in (503, 429):
                print(f"🔄 API rate limited, retrying... attempt {attempt+1}")
                time.sleep(3 * (attempt + 1))
                continue
            return f"API Error {e.response.status_code}: {str(e)}"
        except Exception as e:
            return f"Unexpected Error: {str(e)}"
    
    return "Service unavailable after 3 attempts. Please try again later."

print("Multi-Source Reasoning Active.")

Multi-Source Reasoning Active.


In [8]:
# Test Case
user_question = "what is the admission procedure?"

print(" Vocira is searching across PDFs, Text files, and Web sources...")
print("-" * 50)

# Calling the logic engine we built in Phase 4
answer = ask_vocira(user_question)

print("\n" + "█" * 60)
print("             VOCIRA'S  RESPONSE               ")
print("█" * 60)
print(f"\n{answer}\n")
print("█" * 60)

 Vocira is searching across PDFs, Text files, and Web sources...
--------------------------------------------------
🔍 Searching Knowledge Base...

████████████████████████████████████████████████████████████
             VOCIRA'S  RESPONSE               
████████████████████████████████████████████████████████████

For pre-school admission, there is no written test required. The admission process for primary and secondary school involves an admission test. For more detailed information on the specific steps and requirements of the admission procedure, you can apply online or contact the admissions office at +92-42-3555-1234 or admissions@eliteinternational.edu.pk.

📍 Sources: ..\data\pdf\school_policies.pdf, https://www.educators.edu.pk/, ..\data\text_files\general_info.txt

████████████████████████████████████████████████████████████
